# Ejercicios de Prompt Engineering con Groq

## Configuración Inicial

In [ ]:
from groq import Groq
from IPython.display import display
from IPython.core.display import Markdown

groq_api_key = "Tu_API_Key_Aquí"  # Reemplaza por la API KEY que te generes
client = Groq(api_key=groq_api_key)

## Consultar los modelos que tenemos disponibles

In [ ]:
models = client.models.list()
for model in models.data:
    print(model.id)

## Si queremos un display 'bonito':

In [ ]:
prompt = "Cuentame sobre la historia del Real Madrid y como ganaron 15 champions league"
response = client.chat.completions.create(
    model = "qwen/qwen3-32b",
    messages = [{"role": "user", "content": prompt}],
    temperature = 0.7,
    max_completion_tokens = 500,
    stream = False
)

display(Markdown(response.choices[0].message.content)) # Usamos display(Markdown()) para parsear la respuesta del LLM en Markdown

## Comparando Modelos

Groq ofrece varios modelos. Veamos cómo responden diferentes modelos a la misma pregunta:

In [ ]:
prompt = "Explica qué es la inteligencia artificial en 2 frases cortas"

modelos_a_probar = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "groq/compound-mini"
]

for modelo in modelos_a_probar:
    try:
        print(f"\n{'-'*60}")
        print(f"MODELO: {modelo}")
        print('-'*60)
        
        response = client.chat.completions.create(
            model=modelo,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_completion_tokens=200,
            stream=False
        )
        response_content = response.choices[0].message.content
        display(Markdown(response_content))
        
    except Exception as e:
        print(f"Error con modelo {modelo}: {e}")

# El protocolo system-user-assistant

El protocolo system-user-assistant es la estructura de mensajes que usan los LLMs para entender conversaciones. Es como el "lenguaje" que les indica quién dice qué y con qué propósito.

### System:

Son las instrucciones del director.
- Define cómo debe comportarse el asistente
- Se pone al inicio de la conversación
- El usuario final normalmente no lo ve
- Ejemplos de uso:
    - Establecer personalidad: "Eres un experto en física"
    - Definir restricciones: "Solo responde con código Python, sin explicaciones"
    - Configurar formato: "Responde siempre en formato JSON"
    - Dar contexto: "Eres un asistente de atención al cliente de una empresa de telefonía"
### User:

Es el humano que pregunta
- Las preguntas o inputs del usuario real
- Lo que tú o el cliente escribe
### Assistant: 

Es el LLM que responde
- Las respuestas que genera el modelo
- En conversaciones multi-turno, incluyes las respuestas previas del asistente para mantener contexto

In [ ]:
# Sin system message
print("--- SIN SYSTEM MESSAGE ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "¿Qué es Python?"}
    ],
    temperature=0.7,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

# Con system message: Profesor universitario
print("\n\n--- CON SYSTEM MESSAGE: Profesor Universitario ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "Eres un profesor universitario de informática con 20 años de experiencia. Tus respuestas son técnicas, precisas y académicas."
        },
        {"role": "user", "content": "¿Qué es Python?"}
    ],
    temperature=0.7,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

# Con system message: Poeta
print("\n\n--- CON SYSTEM MESSAGE: Poeta ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "Eres un poeta romántico. Respondes siempre de forma poética y metafórica."
        },
        {"role": "user", "content": "¿Qué es Python?"}
    ],
    temperature=0.7,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

El system message es muy potente porque el modelo le da más peso que a los mensajes de user. Es como la diferencia entre:

- system: "Tu jefe te dice cómo hacer tu trabajo"
- user: "Un cliente te hace una pregunta"

El modelo intenta seguir las instrucciones del system incluso si el user le pide algo diferente.

## Conversaciones Multi-turno

Para mantener una conversación, debes incluir todo el historial de mensajes. El modelo no tiene memoria entre llamadas. Nótese como incluimos en el historial el rol assistant: el LLM y su respuesta.

In [ ]:
# Simulamos una conversación
historial = [
    {"role": "system", "content": "Eres un asistente útil y amigable."},
    {"role": "user", "content": "Hola, me llamo Alonso."}
]

# Primera interacción
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=historial,
    temperature=0.7,
    stream=False
)
respuesta1 = response.choices[0].message.content
print("Asistente:", respuesta1)

# Añadimos la respuesta al historial
historial.append({"role": "assistant", "content": respuesta1})

# Segunda pregunta (el modelo debe recordar el nombre)
historial.append({"role": "user", "content": "¿Cuál es mi nombre?"})
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=historial,
    temperature=0.7,
    stream=False
)
respuesta2 = response.choices[0].message.content
print("\nUsuario: ¿Cuál es mi nombre?")
print("Asistente:", respuesta2)

# Tercera pregunta
historial.append({"role": "assistant", "content": respuesta2})
historial.append({"role": "user", "content": "¿Me puedes contar un chiste sobre programación?"})
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=historial,
    temperature=0.8,
    stream=False
)
respuesta3 = response.choices[0].message.content
print("\nUsuario: ¿Me puedes contar un chiste sobre programación?")
print("Asistente:", respuesta3)

## Controlando la longitud con max_completion_tokens

Este parámetro limita cuántos tokens puede generar el modelo.

In [ ]:
prompt = "Cuéntame sobre la historia de Internet"

# Respuesta muy corta
print("--- max_completion_tokens = 50 ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_completion_tokens=50,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))
print(f"\n(Se generaron {response.usage.completion_tokens} tokens)")

# Respuesta media
print("\n\n--- max_completion_tokens = 200 ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_completion_tokens=200,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))
print(f"\n(Se generaron {response.usage.completion_tokens} tokens)")

# Respuesta larga
print("\n\n--- max_completion_tokens = 500 ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_completion_tokens=500,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))
print(f"\n(Se generaron {response.usage.completion_tokens} tokens)")

## Jugando con Top_P (Nucleus Sampling)

**top_p** controla la diversidad filtrando tokens menos probables:
- **0.1**: Solo considera los tokens más probables (muy conservador)
- **0.9-1.0**: Considera más opciones (más diverso)

Es una alternativa a temperature. Generalmente se usa uno u otro, no ambos al máximo.

In [ ]:
prompt = "Completa esta frase: El secreto de la felicidad es"

# Top_P bajo
print("--- TOP_P = 0.1 (Muy focalizado) ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=1,
    top_p=0.1,
    max_completion_tokens=150,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

# Top_P alto
print("\n--- TOP_P = 0.95 (Más diverso) ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=1,
    top_p=0.95,
    max_completion_tokens=150,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

## Experimentando con Temperature

La **temperature** controla la creatividad del modelo:
- **0.0-0.3**: Respuestas más deterministas y conservadoras
- **0.7-1.0**: Respuestas más creativas y variadas
- **>1.0**: Muy aleatorias (puede producir resultados incoherentes)

In [ ]:
prompt = "Escribe un eslogan creativo para una cafetería"

# Temperature BAJA (más conservador)
print("--- TEMPERATURE = 0.2 (Conservador) ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
    max_completion_tokens=100,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

# Temperature MEDIA
print("\n--- TEMPERATURE = 0.7 (Balanceado) ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_completion_tokens=100,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

# Temperature ALTA (más creativo)
print("\n--- TEMPERATURE = 1.5 (Muy Creativo) ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=1.5,
    max_completion_tokens=100,
    stream=False
)
response_content = response.choices[0].message.content
display(Markdown(response_content))

## Modo Streaming vs No-Streaming

El modo **streaming** va mostrando la respuesta palabra por palabra (como ChatGPT), mientras que el **no-streaming** espera a tener la respuesta completa. Veamos ambos:

In [ ]:
# Modo STREAMING: Muestra token por token
print("\n--- MODO STREAMING ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Dame 3 datos curiosos sobre el espacio"
        }
    ],
    temperature=1,
    max_completion_tokens=500,
    stream=True  # CON streaming
)

# Muestra cada token conforme se genera
for chunk in response:
    print(chunk.choices[0].delta.content or "", end="")

In [ ]:
# Modo NO-STREAMING: Espera la respuesta completa
print("--- MODO NO-STREAMING ---")
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Dame 3 datos curiosos sobre el espacio"
        }
    ],
    temperature=1,
    max_completion_tokens=500,
    stream=False  # NO streaming
)

# La respuesta viene completa
print(response.choices[0].message.content)